<a href="https://colab.research.google.com/github/vorhersager/deep-learning-jax/blob/main/Tutorial_12_Intelligent_Diagnostics_and_Hybrid_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 12: Intelligent Diagnostics and Hybrid-AI for Global Macroeconomic Integrity

**An advanced framework for unsupervised anomaly detection, mathematical explainability, and causal reasoning.**


Instructor: John Sipple `jsipple@gwu.edu`

1 June 2026

---

**Important** To execute the final part with cuasal reasoning you will need to have a Gemini API key. If you don't you will get the following error:
`SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.`

For more information, please see the [Gemini API documentation](https://ai.google.dev/gemini-api/docs/api-key).

---

### **Overview**
Welcome to the reference implementation of the **Intelligent Diagnostics** framework.

In the realm of global finance and development, traditional rule-based auditing systems are failing. They are brittle, require perfect data, and are entirely blind to "unknown unknowns"—novel crisis patterns or sophisticated financial deviations that have never been seen before. Furthermore, when modern "black-box" AI systems flag an anomaly, they provide no reasoning, leaving auditors and policymakers with unactionable alerts.

This notebook demonstrates a paradigm shift. We introduce a unified, hybrid-AI architecture that not only detects unseen anomalies but explains *why* they occurred and traces them back to actionable geopolitical root causes.

### **The Three Pillars of Intelligent Diagnostics**
This solution synthesizes three state-of-the-art domains of artificial intelligence:

1. **Unsupervised Anomaly Detection (JAX/Flax):** Instead of hunting for known historical crises, we use a technique called **Negative Sampling**. By training a high-performance neural network to contrast real-world data against synthetic uniform noise, the model organically learns the complex, multi-dimensional boundary of "Normal" behavior. Anything that falls outside this manifold is flagged.
2. **Mathematical Explainability (Integrated Gradients):** When an anomaly is detected, we cannot accept a black-box alert. We utilize Integrated Gradients—calculating the path integral of the network's gradients from a baseline to the anomaly—to extract the precise mathematical attributions (symptoms) that triggered the flag.
3. **Causal Reasoning (Gemini LLM):** Statistical symptoms are not root causes. We feed the mathematical attributions into Google's Gemini LLM to act as a causal inference engine. Gemini dynamically generates a Judea Pearl-style Directed Acyclic Graph (DAG) to map the symptoms backwards through intermediate crises, ultimately isolating the structural, geopolitical root causes.

### **Solving the World Bank Challenge**
The World Bank operates in an incredibly noisy global environment where macroeconomic data is frequently delayed, incomplete, or manipulated.

By deploying Intelligent Diagnostics, the World Bank can shift from a *reactive* stance to a *proactive* one. This pipeline allows economists to automatically monitor the global manifold, detecting the earliest mathematical tremors of sovereign debt distress, institutional collapse, or hyperinflation. More importantly, the system's causal reasoning engine ensures that every mathematical anomaly is instantly translated into a targeted, actionable policy recommendation (e.g., deploying emergency infrastructure grants or IMF structural adjustment facilities).

### **The Implementation: Live Macroeconomic Discovery**
To prove this architecture in the wild, this notebook bypasses sterile academic datasets.

We will connect directly to the **World Bank API (`wbgapi`)** to pull live, multivariate macroeconomic data (GDP per Capita, Inflation, Life Expectancy, Population Growth, and Unemployment) for over 200 global economies.

**In this interactive notebook, we will:**
* **Stage A:** Fetch live data and use Median Imputation to handle real-world reporting gaps.
* **Stage B & C:** Train a dynamic, regularized JAX neural network to learn the current global economic manifold, visualizing its trained weights.
* **Stage D:** Score every country on Earth to discover the most severe, undocumented macroeconomic anomaly.
* **Stage E:** Run Integrated Gradients to extract the exact feature deviations causing the crisis.
* **Stage F:** Chain two Gemini LLM agents to dynamically generate a massive Causal Knowledge Graph and draft a 120-word executive policy brief for World Bank leadership.

#Environment Setup

**Intelligent Diagnostics: Hybrid-AI for Anomaly Detection & Causal Reasoning**

Before we begin, we must ensure our high-performance computing environment is properly configured. We rely on JAX and Flax for XLA-accelerated neural network operations, Optax for gradient processing, and Google Generative AI (Gemini) for semantic causal reasoning.

This first cell installs the required dependencies and imports the necessary libraries for data processing, model training, and graph visualization.

In [ ]:
# Install required libraries (uncomment if running in a fresh Colab environment)
# !pip install flax optax google-generativeai networkx scikit-learn matplotlib

import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import pairwise_distances_argmin_min
import google.generativeai as genai
import json
import os

# Set global print options for cleaner output
np.set_printoptions(precision=4, suppress=True)
print("Environment setup complete. JAX backend:", jax.default_backend())

#Data Acquisition & Negative SamplingDefining the "Normal" Manifold

In unsupervised anomaly detection, we do not hunt for specific fraud patterns, as that leaves us blind to novel attacks. Instead, we teach the model the strict boundaries of "Normal" behavior.

1. Dataset: We fetch a standard Credit Card Fraud dataset from OpenML. It contains multivariate, continuous features (PCA transformations of transaction data and an 'Amount' feature).

2. Standardization: We compute the mean and variance exclusively on the normal training data to prevent data leakage from anomalies.

3. Negative Sampling: To force the neural network to learn a decision boundary without labeled fraud data, we generate synthetic "negative samples". We do this by uniformly sampling points within the $d$-dimensional bounding box of the normalized training data. The model will learn to classify true historical data as $0$ and these synthetic uniform points as $1$.

In [ ]:
!pip install wbgapi

# Macroeconomic Data Acquisition & Negative Sampling

**Defining the "Normal" Manifold of Global Economies**

For this demonstration, we will pull live data using the World Bank's official API (wbgapi). We will construct a multivariate dataset of global economies based on five critical development indicators: GDP per capita, Inflation, Life Expectancy, CO2 Emissions, and Unemployment.

In unsupervised anomaly detection, we do not provide historical labels of "failed states" or "economic crises". Instead, we teach the model the boundaries of a "Normal" functioning economy in a given year.

1. **Dataset**: Cross-sectional macroeconomic indicators for all global economies using the most recent valid year.

2. **Standardization**: We normalize the features so that vastly different scales (e.g., GDP in thousands vs. Inflation in percentages) are weighted equally by the neural network.

3. **Negative Sampling**: We generate synthetic "negative samples" by uniformly sampling points within the bounds of our global data. The network learns a boundary by contrasting real, historical country profiles against this synthetic noise, allowing it to autonomously identify countries undergoing severe, complex crises without needing predefined rules.

In [ ]:
# Install the official World Bank API client
# !pip install wbgapi

import wbgapi as wb
import pandas as pd
import numpy as np

print("Fetching historical macroeconomic indicators (2015-2023) to guarantee data density...")

# We are defining 5 distinct dimensions to test the multivariate AI
indicators = {
    'NY.GDP.PCAP.CD': 'GDP_Per_Capita_USD',
    'FP.CPI.TOTL.ZG': 'Inflation_Annual_Pct',
    'SP.DYN.LE00.IN': 'Life_Expectancy_Years',
    'SP.POP.GROW': 'Population_Growth_Pct',
    'SL.UEM.TOTL.ZS': 'Unemployment_Pct'
}

# 1. BULLETPROOF FETCH: Pull raw data over an 8-year window.
# This bypasses the API's tendency to drop columns that don't share the exact same latest year.
raw_data = list(wb.data.fetch(list(indicators.keys()), time=range(2015, 2024)))
df_raw = pd.DataFrame(raw_data)

# 2. Average the data over the time window for each country/indicator pair.
# This guarantees a dense matrix and provides a highly stable "baseline" for normal economic behavior.
wb_data = df_raw.groupby(['economy', 'series'])['value'].mean().unstack()

# 3. Rename the columns safely to our human-readable features
valid_indicators = {k: v for k, v in indicators.items() if k in wb_data.columns}
wb_data = wb_data.rename(columns=valid_indicators)

# 4. Fetch country metadata to get human-readable names and join it
economies = wb.economy.DataFrame()['name']
wb_data = wb_data.join(economies).rename(columns={'name': 'Country'})

# 5. Median Imputation to patch any straggling gaps in the 8-year window
feature_names = list(valid_indicators.values())
wb_data[feature_names] = wb_data[feature_names].fillna(wb_data[feature_names].median())

# 6. Drop any rogue rows that lack a valid country name
wb_data = wb_data.dropna(subset=['Country'])

# Extract feature names and values
X_real = wb_data[feature_names].values.astype(np.float32)
country_names = wb_data['Country'].values

print(f"Success! Loaded {len(X_real)} economies across {len(feature_names)} active dimensions.")
print(f"Active Features: {feature_names}")

# --- Normalization ---
# Compute statistics to standardize the real-world data
train_mean = np.mean(X_real, axis=0)
train_std = np.std(X_real, axis=0) + 1e-8 # Add epsilon to prevent divide-by-zero

def normalize(data):
    """Standardizes data using real-world statistics."""
    return (data - train_mean) / train_std

def denormalize(data):
    """Reverts standardized data back to real-world macroeconomic values."""
    return (data * train_std) + train_mean

# Apply normalization
X_real_norm = normalize(X_real)

# --- Negative Sampling ---
# We determine the min and max bounds of the global economic manifold
num_neg_samples = len(X_real_norm) * 2 # Generate double the noise for a robust boundary

# Use nanmin/nanmax and cast to float64 to ensure the numpy generator never overflows
min_vals = np.nanmin(X_real_norm, axis=0).astype(np.float64)
max_vals = np.nanmax(X_real_norm, axis=0).astype(np.float64)

print(f"Generating {num_neg_samples} synthetic 'negative sample' economies...")
np.random.seed(42)

# Draw uniform samples within the operational bounds
X_neg = np.random.uniform(
    low=min_vals,
    high=max_vals,
    size=(num_neg_samples, X_real_norm.shape[1])
).astype(np.float32)

# Combine real data (Label 0: Normal) with synthetic negative data (Label 1: Anomaly)
X_train_combined = np.vstack((X_real_norm, X_neg))
y_train_combined = np.hstack((np.zeros(len(X_real_norm)), np.ones(num_neg_samples)))

# Shuffle the combined dataset to prevent batch bias during gradient descent
shuffle_idx = np.random.permutation(len(X_train_combined))
X_train_combined = X_train_combined[shuffle_idx]
y_train_combined = y_train_combined[shuffle_idx]

print("Data preparation complete. Shape of training tensor:", X_train_combined.shape)

#Exploratory Data Analysis & Visual Inspection

**The Limits of Human Observation**

Before training the Intelligent Diagnostics model, it is critical to visually inspect the macroeconomic manifold.

Below, we generate a **Pair Plot** (a scatter plot matrix) comparing every feature against every other feature. To highlight extreme data points, we calculate the combined Z-score (standard deviations from the mean) across all dimensions and flag the top 3 most statistically extreme countries in the raw data.

Why do this? Scatter plots easily reveal simple, 1D or 2D outliers (e.g., a single country experiencing hyperinflation). However, human visualization fails beyond three dimensions. A country might have perfectly "average" GDP, average inflation, and average unemployment when viewed individually, but the combination of those factors might be highly anomalous. This visual limitation is exactly why we rely on our multi-dimensional JAX neural network to define the true boundary of normal operations.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

print("Generating macroeconomic pair plots and isolating visual outliers...")

# 1. Create a copy of the dataframe specifically for visualization
viz_df = wb_data.copy()

# 2. Simple Heuristic to find the most "visually obvious" extreme outliers
# We calculate the sum of absolute Z-scores across all features to find the raw extremes
z_scores = np.abs((X_real - np.mean(X_real, axis=0)) / (np.std(X_real, axis=0) + 1e-8))
extreme_scores = z_scores.sum(axis=1)

# Grab the indices of the top 3 most extreme countries in the raw data
top_outlier_indices = np.argsort(extreme_scores)[-3:]
outlier_countries = country_names[top_outlier_indices]

# 3. Create a 'Label' column to color-code the scatter plots
viz_df['Label'] = 'Standard Economy'
viz_df.loc[viz_df['Country'].isin(outlier_countries), 'Label'] = 'Extreme Visible Outlier'

# 4. Configure the presentation aesthetics (Dark Mode)
sns.set_theme(style="darkgrid", rc={
    "axes.facecolor": "#0b192c",
    "figure.facecolor": "#0b192c",
    "text.color": "white",
    "axes.labelcolor": "white",
    "xtick.color": "white",
    "ytick.color": "white",
    "grid.color": "#1e3e62"
})

# 5. Generate the Pair Plot
g = sns.pairplot(
    viz_df,
    vars=feature_names,
    hue='Label',
    palette={'Standard Economy': '#00ffd1', 'Extreme Visible Outlier': '#ff4d4d'},
    plot_kws={'alpha': 0.6, 's': 40, 'edgecolor': 'none'},
    diag_kws={'alpha': 0.7, 'edgecolor': 'none'},
    height=2.5
)

g.fig.suptitle("Macroeconomic Manifold: 2D Pairwise Projections", y=1.02, color='white', fontsize=20, fontweight='bold')

# 6. Annotate the exact names of the outlier countries on the scatter plots
for i, y_var in enumerate(feature_names):
    for j, x_var in enumerate(feature_names):
        if i != j: # Only annotate the off-diagonal scatter plots
            ax = g.axes[i, j]
            for idx in top_outlier_indices:
                x_val = viz_df.iloc[idx][x_var]
                y_val = viz_df.iloc[idx][y_var]
                country = viz_df.iloc[idx]['Country']

                # Plot text slightly offset from the dot
                ax.annotate(
                    country,
                    (x_val, y_val),
                    xytext=(5, 5),
                    textcoords='offset points',
                    fontsize=9,
                    color='#ff4d4d',
                    weight='bold',
                    alpha=0.9
                )

# Ensure legend text is white
for text in g._legend.texts:
    text.set_color("white")

plt.show()

print(f"Top visible outliers highlighted: {', '.join(outlier_countries)}")

#JAX Neural Network Definition & Training

**Training the Detector via Contrastive Loss**

Here we define a Multi-Layer Perceptron (MLP) using Flax, JAX's neural network library.
We use JAX's `@jax.jit` (Just-In-Time compilation) decorator to compile our train_step and eval_step functions directly to XLA (Accelerated Linear Algebra). This allows the training loop to run orders of magnitude faster than standard Python execution.

The model optimizes a Binary Cross Entropy (BCE) loss. By contrasting the dense real data against the sparse uniform negative samples, the network naturally carves out a tight, non-linear contour around "normal" operations.

In [ ]:
from typing import Sequence
from flax.training import train_state

# 1. Define the Dynamic Neural Network Architecture using Flax
class DynamicAnomalyDetector(nn.Module):
    # Parameterize the architecture
    hidden_sizes: Sequence[int]
    dropout_rate: float = 0.2

    @nn.compact
    def __call__(self, x, deterministic: bool = True):
        """
        Forward pass.
        deterministic=False triggers Dropout during training.
        deterministic=True bypasses Dropout during evaluation.
        """
        for size in self.hidden_sizes:
            # Apply Dense Layer
            x = nn.Dense(size)(x)
            # Apply Non-linear Activation
            x = nn.relu(x)
            # Apply Dropout (Requires PRNG key if deterministic=False)
            x = nn.Dropout(rate=self.dropout_rate, deterministic=deterministic)(x)

        # Final output layer (single node for binary probability logits)
        x = nn.Dense(1)(x)
        return x

def create_train_state(rng, hidden_sizes, learning_rate):
    """Initializes the dynamic model, parameters, and optimizer state."""
    model = DynamicAnomalyDetector(hidden_sizes=hidden_sizes, dropout_rate=0.2)
    # Initialize parameters by passing a dummy input and setting deterministic=True
    params = model.init(rng, jnp.ones([1, X_train_combined.shape[1]]), deterministic=True)['params']
    tx = optax.adam(learning_rate)
    return model, params, tx, train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

# 2. JIT compile the training step
@jax.jit
def train_step(state, batch_X, batch_y, dropout_key):
    def loss_fn(params):
        # Forward pass: deterministic=False turns ON dropout.
        # We MUST provide the 'dropout' PRNG key here.
        logits = state.apply_fn(
            {'params': params},
            batch_X,
            deterministic=False,
            rngs={'dropout': dropout_key}
        )
        # Compute BCE Loss
        loss = optax.sigmoid_binary_cross_entropy(logits=logits.squeeze(), labels=batch_y).mean()
        return loss

    # Compute loss and gradients
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    # Update weights
    state = state.apply_gradients(grads=grads)
    return state, loss

print("Initializing dynamic model and starting training loop...")

# Define our new dynamic architecture: 64 nodes -> 32 nodes -> 16 nodes
architecture_array = [64, 32, 16]

# Manage JAX random state explicitly
rng = jax.random.PRNGKey(0)
rng, init_rng, dropout_rng = jax.random.split(rng, 3)

# Create the training state
model, params, tx, state = create_train_state(init_rng, architecture_array, learning_rate=1e-3)

epochs = 30
batch_size = 64
train_losses = []

# --- Training Loop ---
for epoch in range(epochs):
    epoch_loss = 0.0
    num_batches = 0

    # Iterate through batches
    for i in range(0, len(X_train_combined), batch_size):
        batch_X = X_train_combined[i:i+batch_size]
        batch_y = y_train_combined[i:i+batch_size]

        if len(batch_X) < 2: continue

        # CRITICAL: Split the dropout key for every batch to ensure different neurons are dropped
        dropout_rng, step_dropout_key = jax.random.split(dropout_rng)

        # Execute train step
        state, loss = train_step(state, batch_X, batch_y, step_dropout_key)
        epoch_loss += loss
        num_batches += 1

    # Track average loss
    if num_batches > 0:
        avg_train_loss = epoch_loss / num_batches
        train_losses.append(avg_train_loss)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d} | Train Loss (vs Negative Samples): {avg_train_loss:.4f}")

# Plot Training Convergence
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss (Learning Manifold)', color='#00ffd1', linewidth=2)
plt.title(f"Network Convergence: Layer Topology {architecture_array}")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross Entropy Loss")
plt.gca().set_facecolor('#0b192c')
plt.gcf().set_facecolor('#0b192c')
plt.gca().tick_params(colors='white')
plt.gca().xaxis.label.set_color('white')
plt.gca().yaxis.label.set_color('white')
plt.title(f"Network Convergence: Layer Topology {architecture_array}", color='white')
plt.grid(True, alpha=0.2, color='#ffffff')
plt.legend(facecolor='#1e3e62', edgecolor='#00ffd1', labelcolor='white')
plt.show()

#Visualizing the Trained Weight Manifold

**Peeking Inside the Black Box**

To prove that our Intelligent Diagnostics model is not an incomprehensible black box, we can directly map its internal logic.

The function below extracts the exact, trained mathematical weights (params) directly from the JAX tensor state. To make the complex topology digestible, we visualize a representative "slice" of the network (capped at 10 nodes per layer).

* Line Thickness: Represents the absolute magnitude (importance) of the learned connection.

* Blue/Cyan Edges: Positive weights. These features actively push the transaction toward being classified as an "Anomaly."

* Red Edges: Negative weights. These features actively suppress the anomaly score, pulling the transaction back toward "Normal."

This provides a clear, visual audit trail of how information flows through the decision boundary.

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

def plot_trained_architecture(state, input_dim):
    """
    Extracts live weights from the JAX train state and generates a dynamic
    network graph. Magnitude dictates thickness; Sign dictates color.
    """
    G = nx.DiGraph()

    # 1. Extract the learned weight matrices (kernels) from the Flax parameters
    # Flax automatically names sequential compact layers as 'Dense_0', 'Dense_1', etc.
    params = state.params
    dense_keys = [k for k in params.keys() if k.startswith('Dense_')]

    # Sort keys to ensure we read the network from Input -> Output
    dense_keys.sort(key=lambda x: int(x.split('_')[1]))

    # Extract the weight matrices. JAX tensors are converted to standard numpy arrays for plotting.
    weight_matrices = [np.array(params[k]['kernel']) for k in dense_keys]

    # Determine layer sizes from the shapes of the weight matrices
    layers = [input_dim] + [w.shape[1] for w in weight_matrices]

    layer_names = ['Input\n(Features)']
    for i in range(1, len(weight_matrices)):
        layer_names.append(f"Hidden {i}\n(Dense+Dropout)")
    layer_names.append('Output\n(Anomaly Prob)')

    pos = {}

    # 2. Add Nodes to the Graph
    for i, size in enumerate(layers):
        # Cap visual nodes to a maximum of 10 per layer to prevent visual clutter
        nodes_to_draw = min(size, 10)
        for j in range(nodes_to_draw):
            node_id = f"L{i}_N{j}"
            G.add_node(node_id)
            # Center the nodes vertically
            pos[node_id] = (i, j - nodes_to_draw / 2)

    # 3. Add Edges with Trained Weights
    edges = []
    edge_colors = []
    edge_widths = []

    # Find the global maximum absolute weight for normalizing line thickness
    global_max_weight = max([np.max(np.abs(w)) for w in weight_matrices])

    for i, w_mat in enumerate(weight_matrices):
        nodes_in = min(w_mat.shape[0], 10)
        nodes_out = min(w_mat.shape[1], 10)

        for src_idx in range(nodes_in):
            for dst_idx in range(nodes_out):
                val = float(w_mat[src_idx, dst_idx])
                src_node = f"L{i}_N{src_idx}"
                dst_node = f"L{i+1}_N{dst_idx}"

                G.add_edge(src_node, dst_node, weight=val)
                edges.append((src_node, dst_node))

                # --- Map Sign to Color ---
                # Using Cyan/Blue (#00ffd1) for positive, Red (#ff4d4d) for negative
                if val > 0:
                    color = '#00ffd1' # Positive activation
                else:
                    color = '#ff4d4d' # Negative suppression

                # --- Map Magnitude to Thickness & Transparency ---
                # Normalize weight against the global max
                normalized_weight = abs(val) / (global_max_weight + 1e-8)

                # Base thickness + scaled thickness
                width = 0.5 + (normalized_weight * 4.0)

                # Add alpha (transparency) so tiny weights fade into the background
                alpha = 0.15 + (normalized_weight * 0.85)
                rgba_color = mcolors.to_rgba(color, alpha=alpha)

                edge_colors.append(rgba_color)
                edge_widths.append(width)

    # 4. Plotting Configuration
    plt.figure(figsize=(14, 7))

    # Draw edges with calculated colors and widths
    nx.draw_networkx_edges(G, pos, edgelist=edges, edge_color=edge_colors,
                           width=edge_widths, arrows=False)

    # Draw nodes: Input/Output are solid, Hidden layers are orange
    node_colors_list = []
    for node in G.nodes():
        layer_idx = int(node.split('_')[0][1:])
        if layer_idx == 0 or layer_idx == len(layers) - 1:
            node_colors_list.append('#00ffd1') # Input/Output
        else:
            node_colors_list.append('#ffa64d') # Hidden

    nx.draw_networkx_nodes(G, pos, node_size=100, node_color=node_colors_list,
                           edgecolors='#ffffff', linewidths=1)

    # Add dynamic Layer Titles
    for i, name in enumerate(layer_names):
        plt.text(i, 6, f"{name}\n(Dim: {layers[i]})", ha='center',
                 fontsize=10, fontweight='bold', color='white',
                 bbox=dict(facecolor='#1e3e62', alpha=0.9, edgecolor='#00ffd1', boxstyle='round,pad=0.5'))

    # Legend for the presentation
    plt.text(-0.5, -6, "■ Positive Weight (Drives toward Anomaly)", color='#00ffd1', fontsize=12, fontweight='bold')
    plt.text(-0.5, -6.5, "■ Negative Weight (Suppresses Anomaly flag)", color='#ff4d4d', fontsize=12, fontweight='bold')
    plt.text(-0.5, -7, "Line Thickness = Weight Magnitude", color='#b0bec5', fontsize=12, fontstyle='italic')

    plt.title(f"Live Trained Weight Manifold (Representative Slices)", pad=40, color='white', fontsize=16, fontweight='bold')
    plt.gca().set_facecolor('#0b192c')
    plt.gcf().set_facecolor('#0b192c')
    plt.axis('off')

    # Set y-limits to ensure labels aren't cut off
    plt.ylim(-8, 7)
    plt.show()

# Execute plot using the LIVE trained state parameters
plot_trained_architecture(state, input_dim=X_real.shape[1])

#Global Anomaly Scoring & Discovery

**Evaluating the Unsupervised Manifold**

Because we are using live macroeconomic data, we do not have historical "labels" of which economies are currently in crisis. To evaluate our model, we pass every country through the trained network to calculate its Anomaly Score (from 0.0 to 1.0).

* Score near 0.0: The country sits squarely in the dense center of the "Normal" economic manifold.

* Score near 1.0: The country is exhibiting extreme deviations across multiple dimensions simultaneously, pushing it into the territory of our synthetic negative samples.

Below, we score all global economies and plot the Top 10 Most Anomalous countries against 10 of the most "Normal" baseline countries.

In [ ]:
print("Scoring all global economies through the trained manifold...")

# 1. Forward Pass on Real Data
# CRITICAL: We must pass deterministic=True to bypass the Dropout layers during evaluation
logits_real = state.apply_fn({'params': state.params}, X_real_norm, deterministic=True)

# Convert logits to probabilities via sigmoid activation
probs_real = jax.nn.sigmoid(logits_real).squeeze()

# 2. Map scores to countries and sort
results_df = pd.DataFrame({
    'Country': country_names,
    'Anomaly_Score': np.array(probs_real) # Convert JAX array to standard Numpy
})

# Sort economies from most anomalous (highest score) to most normal (lowest score)
results_df = results_df.sort_values(by='Anomaly_Score', ascending=False).reset_index(drop=True)

# 3. Extract the extremes for visualization
top_k = 10
# The 10 countries with the highest anomaly probabilities
most_anomalous = results_df.head(top_k).copy()
most_anomalous['Class'] = 'Anomalous (High Deviation)'

# The 10 countries with the lowest anomaly probabilities (most "standard" economies)
most_normal = results_df.tail(top_k).copy()
most_normal['Class'] = 'Normal (Baseline Manifold)'

# Combine and sort for the bar chart
plot_data = pd.concat([most_normal, most_anomalous]).sort_values(by='Anomaly_Score', ascending=True)

# 4. Plotting the Results
plt.figure(figsize=(12, 8))

# Assign colors based on the class
colors = ['#00ffd1' if c == 'Normal (Baseline Manifold)' else '#ff4d4d' for c in plot_data['Class']]

# Create horizontal bar chart
bars = plt.barh(plot_data['Country'], plot_data['Anomaly_Score'], color=colors, edgecolor='none')

# Add score labels to the end of the bars
for bar in bars:
    width = bar.get_width()
    # Adjust text positioning based on bar length
    if width > 0.5:
        plt.text(width - 0.05, bar.get_y() + bar.get_height()/2, f'{width:.3f}',
                 va='center', ha='right', color='#0b192c', fontweight='bold', fontsize=10)
    else:
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.3f}',
                 va='center', ha='left', color='white', fontweight='bold', fontsize=10)

# Presentation Formatting
plt.title("Intelligent Diagnostics: Global Macroeconomic Anomaly Scores", color='white', fontsize=16, pad=20)
plt.xlabel("Anomaly Probability (0.0 = Normal, 1.0 = Extreme Outlier)", color='white', fontsize=12)
plt.gca().set_facecolor('#0b192c')
plt.gcf().set_facecolor('#0b192c')
plt.gca().tick_params(colors='white', labelsize=11)
plt.gca().xaxis.label.set_color('white')
plt.xlim(0, 1.05)
plt.grid(axis='x', alpha=0.2, color='#ffffff')

# Custom Legend
import matplotlib.patches as mpatches
cyan_patch = mpatches.Patch(color='#00ffd1', label='Baseline "Normal" Economies')
red_patch = mpatches.Patch(color='#ff4d4d', label='Flagged Anomalies')
plt.legend(handles=[cyan_patch, red_patch], facecolor='#1e3e62', edgecolor='none', labelcolor='white', loc='lower right')

plt.tight_layout()
plt.show()

# Save the top anomaly index for the Integrated Gradients step later
top_anomaly_country = most_anomalous.iloc[0]['Country']
top_anomaly_idx = np.where(country_names == top_anomaly_country)[0][0]

print("\n--- TOP 3 FLAGGED ANOMALIES ---")
for i in range(3):
    print(f"{i+1}. {most_anomalous.iloc[i]['Country']} (Score: {most_anomalous.iloc[i]['Anomaly_Score']:.4f})")

#Integrated Gradients Explainability

**From Black Box to Audit Trail**

When an anomaly is flagged, auditors need to know why. We use Integrated Gradients (IG) to compute feature attributions. IG works by taking the path integral of the model's gradients from a baseline state to the current anomalous input state.

Mathematically, the attribution for feature $i$ is calculated as:

$$IG_i(x) = (x_i - x^\prime_i) \times \int_{\alpha=0}^{1} \frac{\partial F(x^\prime + \alpha(x - x^\prime))}{\partial x_i} d\alpha$$

Where $x$ is the anomalous point, and $x^\prime$ is the baseline. For the baseline, we locate the nearest normal point in the training set. This is crucial: it shows the auditor exactly what specific feature deviations pushed the transaction over the decision boundary.

The output is a precise audit trail showing exactly which macroeconomic deviations pushed the country over the AI's decision boundary, complete with the raw observed data versus the expected baseline.

In [ ]:
def integrated_gradients(state, input_point, baseline_point, steps=50):
    """Computes Integrated Gradients for a JAX/Flax model with Dropout."""
    alphas = jnp.linspace(0.0, 1.0, steps)
    path = baseline_point + alphas[:, None] * (input_point - baseline_point)

    def model_prob_fn(x):
        # CRITICAL: deterministic=True bypasses Dropout.
        # We need the full, stable network to compute accurate gradient paths.
        logits = state.apply_fn({'params': state.params}, x, deterministic=True)
        return jax.nn.sigmoid(logits)[0]

    # Vectorize the gradient computation across the alpha interpolation path
    grad_fn = jax.vmap(jax.grad(model_prob_fn))
    grads = grad_fn(path)

    # Approximate the integral (Trapezoidal rule)
    avg_grads = jnp.mean(grads, axis=0)
    ig_attributions = (input_point - baseline_point) * avg_grads
    return ig_attributions

print(f"Analyzing Top Anomaly: {top_anomaly_country}")

# 1. Identify the Anomaly's exact coordinates in the original dataset
anomalous_pt_norm = X_real_norm[top_anomaly_idx]
anomalous_pt_raw = X_real[top_anomaly_idx]

# 2. Build the "Normal Reference Pool"
# We extract the 15 countries with the lowest anomaly scores to act as our baseline manifold
normal_countries = most_normal['Country'].values
# Find their original indices in the main X_real tensor
normal_indices_original = [np.where(country_names == c)[0][0] for c in normal_countries]

X_normal_pool_norm = X_real_norm[normal_indices_original]
X_normal_pool_raw = X_real[normal_indices_original]

# 3. Find the Nearest Normal Neighbor (Euclidean distance on normalized features)
distances = pairwise_distances_argmin_min([anomalous_pt_norm], X_normal_pool_norm)
baseline_idx_in_pool = distances[0][0]
baseline_original_idx = normal_indices_original[baseline_idx_in_pool]
baseline_country = country_names[baseline_original_idx]

baseline_pt_norm = X_normal_pool_norm[baseline_idx_in_pool]
baseline_pt_raw = X_normal_pool_raw[baseline_idx_in_pool]

print(f"Nearest Normal Baseline Identified: {baseline_country}\n")

# 4. Compute Attributions
attributions = integrated_gradients(state, anomalous_pt_norm, baseline_pt_norm)

# --- 5. Data Visualization Preparation ---
# Sort features by their absolute attribution magnitude
sort_idx = np.argsort(np.abs(attributions))
sorted_features = [feature_names[i] for i in sort_idx]
sorted_attrs = attributions[sort_idx]
sorted_obs = anomalous_pt_raw[sort_idx]
sorted_exp = baseline_pt_raw[sort_idx]

# Save top symptoms for the Gemini prompt in the next cell
symptoms_dict = {}
for i in range(-1, -4, -1): # Grab top 3 from the end of the sorted array
    feat = sorted_features[i]
    symptoms_dict[feat] = {
        "Attribution": float(sorted_attrs[i]),
        "Observed": float(sorted_obs[i]),
        "Expected_Normal": float(sorted_exp[i])
    }

# --- 6. Plotting the Explainability Report ---
plt.figure(figsize=(14, 7))

# Cyan for positive attributions (drives anomaly), Red for negative (suppresses anomaly)
colors = ['#00ffd1' if a > 0 else '#ff4d4d' for a in sorted_attrs]

bars = plt.barh(sorted_features, sorted_attrs, color=colors, edgecolor='none', height=0.6)

# Overlay the Expected vs Observed Data text directly onto the chart
max_attr = np.max(np.abs(sorted_attrs))

for i, bar in enumerate(bars):
    width = bar.get_width()
    obs = sorted_obs[i]
    exp = sorted_exp[i]

    # Format the numbers cleanly
    if 'GDP' in sorted_features[i]:
        text = f"Observed: ${obs:,.0f}  |  Expected Normal: ${exp:,.0f}"
    else:
        text = f"Observed: {obs:.1f}  |  Expected Normal: {exp:.1f}"

    # Position text dynamically based on the bar's direction
    if width > 0:
        plt.text(width + (max_attr * 0.02), bar.get_y() + bar.get_height()/2, text,
                 va='center', color='white', fontsize=11, fontweight='bold')
    else:
        plt.text(width - (max_attr * 0.02), bar.get_y() + bar.get_height()/2, text,
                 va='center', ha='right', color='white', fontsize=11, fontweight='bold')

# Chart Styling
plt.title(f"IG Feature Attributions: {top_anomaly_country} vs. {baseline_country} (Baseline)", color='white', fontsize=16, pad=20)
plt.xlabel("Attribution Score (Impact on Anomaly Probability)", color='white', fontsize=12)

# Aesthetic Dark Mode adjustments
plt.gca().set_facecolor('#0b192c')
plt.gcf().set_facecolor('#0b192c')
plt.gca().tick_params(colors='white', labelsize=12)
plt.gca().xaxis.label.set_color('white')

# Add a subtle vertical line at 0
plt.axvline(x=0, color='#ffffff', linewidth=1.5, alpha=0.5)

# Expand X-limits slightly to ensure the overlaid text fits on the screen
plt.xlim(-max_attr * 1.5, max_attr * 1.5)

# Hide top and right spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['left'].set_color('#ffffff')
plt.gca().spines['bottom'].set_color('#ffffff')

plt.tight_layout()
plt.show()

#Gemini Causal Reasoning & Geopolitical Action

**From Statistical Symptoms to World Bank Interventions**

Integrated Gradients told us what macroeconomic indicators triggered the anomaly. Now, we use the Gemini 1.5 Pro API to figure out why this is happening and how to fix it.

We pass the mathematically extracted symptoms into Gemini, instructing it to act as an expert World Bank economist. It is tasked with:

1. Constructing a Macroeconomic Knowledge Graph that maps the observed statistical symptoms to intermediate domestic crises, and ultimately traces them back to systemic root causes.

2. Ensuring the identified root causes are addressable via geopolitical actions or World Bank financial instruments (e.g., IMF structural adjustments, governance grants, infrastructure funding).

3. Performing a reverse graph traversal to generate an executive summary detailing exactly how the crisis unfolded and where the World Bank should intervene.

In [ ]:
import os
import json
import google.generativeai as genai
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, HTML
from google.colab import userdata


# Selectthe Gemini model
GEMINI_MODEL = "gemini-3.1-pro-preview"
# Configure the Gemini API
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

genai.configure(api_key=GOOGLE_API_KEY)
# Using the pro model for complex causal reasoning and strict JSON compliance
model = genai.GenerativeModel(GEMINI_MODEL)

def step_one_generate_causal_dag(country_name, symptoms):
    """STEP 1: LLM generates the Judea Pearl-style causal graph dynamically."""
    print(f"Step 1: Generating Judea Pearl-style Causal DAG for {country_name}...")

    prompt_1 = f"""
    You are an expert in macroeconomic causal inference (Judea Pearl style).
    Our JAX deep learning model has flagged {country_name} as a severe global outlier.
    Integrated Gradients isolated the following specific macroeconomic symptoms and their deviations:

    {json.dumps(symptoms, indent=2)}

    Construct a sprawling, complex Causal Directed Acyclic Graph (DAG) containing at least 20 nodes.
    - Start with World Bank actionable root causes (e.g., Geopolitical Shocks, Debt Traps).
    - Map the probabilistic downstream effects through intermediate economic variables.
    - End the graph precisely at the observed statistical symptoms provided above (they must be leaf nodes).

    Output STRICTLY as a JSON object with two keys:
    'nodes': A list of all node string names.
    'edges': A list of lists, where each inner list represents a causal directional edge ["Cause Node", "Effect Node"].
    """

    try:
        response = model.generate_content(
            prompt_1,
            generation_config={"response_mime_type": "application/json"}
        )
        graph_data = json.loads(response.text)
        print(f"  -> Success! Generated graph with {len(graph_data['nodes'])} nodes and {len(graph_data['edges'])} edges.")
        return graph_data
    except Exception as e:
        raise RuntimeError(f"API Failed. Please ensure your GOOGLE_API_KEY is active for Step 1. Error: {e}")

def step_two_generate_summary(country_name, symptoms, causal_graph):
    """STEP 2: LLM traverses the generated graph to write the executive summary."""
    print(f"Step 2: Traversing the Causal DAG to formulate World Bank policy...")

    prompt_2 = f"""
    You are the Chief AI Economist for the World Bank.
    Review the following Causal Directed Acyclic Graph (DAG) mapping a crisis in {country_name}:

    {json.dumps(causal_graph, indent=2)}

    The final observed statistical symptoms at the end of this causal chain are:
    {json.dumps(symptoms, indent=2)}

    Task: Perform a reverse graph traversal from the observed symptoms back to the root causes.
    Write an executive summary detailing the genesis of the crisis, tracing the specific pathways in the graph, and recommend a high-level World Bank intervention.

    CRITICAL CONSTRAINT: The summary MUST be EXACTLY 120 words or less. Do not use JSON, return pure text.
    """

    try:
        response = model.generate_content(prompt_2)
        print("  -> Success! Executive summary generated.")
        return response.text
    except Exception as e:
        raise RuntimeError(f"API Failed. Please ensure your GOOGLE_API_KEY is active for Step 2. Error: {e}")

# ==========================================
# EXECUTE THE TWO-STEP PIPELINE
# ==========================================
# 1. Generate the Graph
causal_dag = step_one_generate_causal_dag(top_anomaly_country, symptoms_dict)

# 2. Generate the Summary based on the Graph
executive_summary = step_two_generate_summary(top_anomaly_country, symptoms_dict, causal_dag)

# ==========================================
# DYNAMIC GRAPH VISUALIZATION
# ==========================================
def plot_complex_causal_graph(graph_data, highlight_symptoms):
    """Plots the dynamically generated DAG with high-contrast text rendering."""
    G = nx.DiGraph()
    G.add_nodes_from(graph_data['nodes'])
    G.add_edges_from(graph_data['edges'])

    plt.figure(figsize=(22, 14))

    # Use spring layout but force high separation for complex generated graphs
    pos = nx.spring_layout(G, seed=42, k=2.0, iterations=150)

    # Dynamically calculate node types using Graph Theory (In-Degree / Out-Degree)
    root_nodes = [n for n, d in G.in_degree() if d == 0]

    # Use fuzzy matching for leaf nodes in case the LLM slightly altered the symptom strings
    leaf_nodes = []
    for node in G.nodes():
        for symptom in highlight_symptoms:
            if symptom.lower() in node.lower() or node.lower() in symptom.lower():
                leaf_nodes.append(node)
                break

    intermediate = [n for n in G.nodes() if n not in root_nodes and n not in leaf_nodes]

    # Draw Nodes
    nx.draw_networkx_nodes(G, pos, nodelist=root_nodes, node_color='#ff4d4d', node_shape='s', node_size=3500, edgecolors='white', linewidths=2.5)
    nx.draw_networkx_nodes(G, pos, nodelist=intermediate, node_color='#ffa64d', node_size=2000, alpha=0.9, edgecolors='#ffffff', linewidths=1.5)
    nx.draw_networkx_nodes(G, pos, nodelist=leaf_nodes, node_color='#00ffd1', node_shape='d', node_size=3500, edgecolors='white', linewidths=2.5)

    # Draw Edges
    nx.draw_networkx_edges(G, pos, edge_color='#667788', arrows=True, arrowsize=20, width=2.0, min_source_margin=20, min_target_margin=20, alpha=0.8)

    # High Contrast Labels
    labels = {n: n.replace(' ', '\n') for n in G.nodes()}
    bbox_props = dict(boxstyle="round,pad=0.3", fc="#0b192c", ec="#00ffd1", lw=1, alpha=0.9)
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=10, font_family="sans-serif",
                            font_color='white', font_weight='bold', bbox=bbox_props)

    # Presentation Styling & Legend
    plt.title(f"Judea Pearl Causal DAG: {top_anomaly_country}", color='white', fontsize=24, pad=20, fontweight='bold')
    plt.gca().set_facecolor('#0b192c')
    plt.gcf().set_facecolor('#0b192c')
    plt.axis('off')

    import matplotlib.lines as mlines
    red_patch = mlines.Line2D([], [], color='#ff4d4d', marker='s', linestyle='None', markersize=12, label='Exogenous / Structural Root Cause')
    orange_patch = mlines.Line2D([], [], color='#ffa64d', marker='o', linestyle='None', markersize=12, label='Intermediate Probabilistic Effect')
    cyan_patch = mlines.Line2D([], [], color='#00ffd1', marker='d', linestyle='None', markersize=12, label='Statistical Symptom (IG Target)')

    plt.legend(handles=[red_patch, orange_patch, cyan_patch], facecolor='#1e3e62', edgecolor='#00ffd1', labelcolor='white', loc='lower left', fontsize=14, borderpad=1)

    plt.tight_layout()
    plt.show()

# Extract symptom keys for dynamic highlighting
symptom_names = list(symptoms_dict.keys())

# Render the dynamic DAG
plot_complex_causal_graph(causal_dag, symptom_names)

# ==========================================
# LARGE FONT HTML DASHBOARD
# ==========================================
html_output = f"""
<div style="background-color: #1e3e62; padding: 30px; border-radius: 12px; border: 3px solid #00ffd1; box-shadow: 0px 10px 30px rgba(0,0,0,0.5); max-width: 1000px; margin: 20px auto; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
    <h2 style="color: #00ffd1; margin-top: 0; font-size: 28px; border-bottom: 2px solid rgba(0, 255, 209, 0.3); padding-bottom: 10px;">
        <span style="font-size: 32px;">🌍</span> World Bank Executive Directive: {top_anomaly_country}
    </h2>
    <p style="color: #ffffff; font-size: 24px; line-height: 1.6; font-weight: 400; text-align: justify;">
        {executive_summary}
    </p>
    <div style="margin-top: 20px; text-align: right;">
        <span style="background-color: #ff4d4d; color: white; padding: 6px 18px; border-radius: 20px; font-size: 16px; font-weight: bold; letter-spacing: 1px;">ACTIONABLE INTELLIGENCE GENERATED</span>
    </div>
</div>
"""

display(HTML(html_output))